# HTMX Integration Example

This notebook demonstrates using cjm-tqdm-capture with HTMX for server-side rendering and SSE.

In [1]:
from fasthtml.common import *
from starlette.responses import StreamingResponse, JSONResponse
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_sizes
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.CUPCAKE)

# Initialize
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Sample tasks
def file_processing_task():
    from tqdm import tqdm
    import time
    
    files = [f"file_{i}.txt" for i in range(50)]
    processed = []
    
    for file in tqdm(files, desc="Processing files"):
        time.sleep(0.05)  # Simulate file processing
        processed.append(file)
    
    return processed

def data_analysis_task():
    from tqdm import tqdm
    import time
    
    # Load data
    for _ in tqdm(range(30), desc="Loading data"):
        time.sleep(0.02)
    
    # Analyze
    for _ in tqdm(range(50), desc="Analyzing"):
        time.sleep(0.03)
    
    # Generate report
    for _ in tqdm(range(20), desc="Generating report"):
        time.sleep(0.02)
    
    return "Analysis complete"

In [5]:
# HTMX-powered UI
@rt("/")
def home():
    return create_test_page(
        "HTMX Progress Demo",
        Div(
            H1("HTMX + SSE Progress Tracking", cls=combine_classes(font_size._2xl, font_weight.bold, m.b(6))),
            
            # Task selection
            Div(
                H2("Select Task", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                Div(
                    Button(
                        "Process Files",
                        hx_post="/start-task",
                        hx_vals='{"task_type": "files"}',
                        hx_target="#progress-container",
                        hx_swap="innerHTML",
                        cls=combine_classes(btn, btn_colors.primary, m.r(2))
                    ),
                    Button(
                        "Run Analysis",
                        hx_post="/start-task",
                        hx_vals='{"task_type": "analysis"}',
                        hx_target="#progress-container",
                        hx_swap="innerHTML",
                        cls=combine_classes(btn, btn_colors.secondary)
                    ),
                    cls=str(m.b(4))
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress container
            Div(id="progress-container", cls=str(m.b(6))),
            
            # Active jobs
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
                Div(
                    id="jobs-container",
                    hx_get="/jobs-list",
                    hx_trigger="load, every 2s",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        ),
        # Include HTMX SSE extension
        Script(src="https://unpkg.com/htmx.org@1.9.10/dist/ext/sse.js"),
        Script("""
            // Helper function to handle SSE messages
            document.body.addEventListener('htmx:sseMessage', function(evt) {
                if (evt.detail.type === 'progress') {
                    const data = JSON.parse(evt.detail.data);
                    const container = evt.detail.elt;
                    
                    // Update progress bars
                    if (data.bars) {
                        for (const [barId, bar] of Object.entries(data.bars)) {
                            const progressEl = container.querySelector(`#progress-${barId}`);
                            const textEl = container.querySelector(`#text-${barId}`);
                            
                            if (progressEl) {
                                progressEl.value = bar.pct;
                                if (textEl) {
                                    textEl.textContent = `${bar.desc}: ${bar.pct.toFixed(1)}% (${bar.cur}/${bar.tot})`;
                                }
                            }
                        }
                    }
                    
                    // Update overall progress
                    const overallEl = container.querySelector('#overall-progress');
                    const overallTextEl = container.querySelector('#overall-text');
                    if (overallEl) {
                        overallEl.value = data.progress;
                        if (overallTextEl) {
                            overallTextEl.textContent = `Overall: ${data.progress.toFixed(1)}%`;
                        }
                    }
                    
                    // Handle completion
                    if (data.completed) {
                        const statusEl = container.querySelector('#job-status');
                        if (statusEl) {
                            statusEl.textContent = 'Task completed!';
                            statusEl.className = '""" + combine_classes(alert, alert_colors.success) + """';
                        }
                        
                        // Refresh after completion
                        setTimeout(() => {
                            htmx.trigger('#jobs-container', 'refresh');
                        }, 1000);
                    }
                }
            });
        """)
    )

In [6]:
# HTMX endpoints
@rt("/start-task", methods=["POST"])
async def start_task(request):
    form = await request.form()
    task_type = form.get('task_type', 'files')
    
    job_id = str(uuid.uuid4())
    
    # Select task based on type
    if task_type == 'analysis':
        task = data_analysis_task
        task_name = "Data Analysis"
    else:
        task = file_processing_task
        task_name = "File Processing"
    
    # Start the job with appropriate throttling
    runner.start(
        job_id,
        task,
        patch_kwargs={
            "min_delta_pct": 2,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return HTMX SSE component with manual JavaScript SSE handling
    return Div(
        H3(f"{task_name} Progress", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(4))),
        
        # Status
        Div(
            f"Job ID: {job_id[:8]}...",
            id="job-status",
            cls=combine_classes(alert, alert_colors.info, m.b(4))
        ),
        
        # Overall progress
        Div(
            P("Overall: 0%", id="overall-text", cls=str(font_weight.bold)),
            Progress(
                id="overall-progress",
                max="100",
                value="0",
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
        ),
        
        # Individual progress bars container
        Div(
            id="progress-bars",
            cls=str(space.y(2))
        ),
        
        # Use JavaScript SSE instead of HTMX SSE extension
        Script(f"""
            // Create progress bar elements dynamically
            const barsContainer = document.getElementById('progress-bars');
            const jobId = '{job_id}';
            
            // Pre-create expected progress bars
            if ('{task_type}' === 'analysis') {{
                const bars = [
                    {{id: 'bar-1', desc: 'Loading data'}},
                    {{id: 'bar-2', desc: 'Analyzing'}},
                    {{id: 'bar-3', desc: 'Generating report'}}
                ];
                bars.forEach(bar => {{
                    const div = document.createElement('div');
                    div.innerHTML = `
                        <p id="text-${{bar.id}}" class="{font_size.sm}">${{bar.desc}}: 0% (0/0)</p>
                        <progress id="progress-${{bar.id}}" class="{combine_classes(progress, progress_colors.accent, w.full)}" value="0" max="100"></progress>
                    `;
                    barsContainer.appendChild(div);
                }});
            }} else {{
                const div = document.createElement('div');
                div.innerHTML = `
                    <p id="text-bar-1" class="{font_size.sm}">Processing files: 0% (0/0)</p>
                    <progress id="progress-bar-1" class="{combine_classes(progress, progress_colors.accent, w.full)}" value="0" max="100"></progress>
                `;
                barsContainer.appendChild(div);
            }}
            
            // Connect SSE
            const sse = new EventSource(`/sse/${{jobId}}`);
            
            sse.onmessage = (event) => {{
                const data = JSON.parse(event.data);
                
                // Update overall progress
                document.getElementById('overall-progress').value = data.progress || 0;
                document.getElementById('overall-text').textContent = `Overall: ${{(data.progress || 0).toFixed(1)}}%`;
                
                // Update individual bars
                if (data.bars) {{
                    for (const [barId, bar] of Object.entries(data.bars)) {{
                        const progressEl = document.getElementById(`progress-${{barId}}`);
                        const textEl = document.getElementById(`text-${{barId}}`);
                        
                        if (progressEl) {{
                            progressEl.value = bar.pct;
                            if (textEl) {{
                                textEl.textContent = `${{bar.desc}}: ${{bar.pct.toFixed(1)}}% (${{bar.cur}}/${{bar.tot}})`;
                            }}
                        }}
                    }}
                }}
                
                // Handle completion
                if (data.completed) {{
                    document.getElementById('job-status').textContent = 'Task completed!';
                    document.getElementById('job-status').className = '{combine_classes(alert, alert_colors.success)}';
                    sse.close();
                    
                    // Trigger HTMX refresh
                    setTimeout(() => {{
                        htmx.trigger('#jobs-container', 'refresh');
                    }}, 1000);
                }}
            }};
            
            sse.onerror = () => {{
                console.log('SSE connection closed');
                sse.close();
            }};
        """),
        
        cls=combine_classes(card, bg_dui.base_200, p(6))
    )

@rt("/jobs-list")
def jobs_list():
    jobs = monitor.all()
    
    if not jobs:
        return P("No active jobs", cls=str(text_color.gray(500)))
    
    return Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Bars")
            )
        ),
        Tbody(
            *[
                Tr(
                    Td(job_id[:8] + "..."),
                    Td(f"{job['overall_progress']:.1f}%"),
                    Td(
                        Span(
                            "Complete" if job['completed'] else "Running",
                            cls=combine_classes(badge, badge_colors.success) if job['completed'] else combine_classes(badge, badge_colors.warning)
                        )
                    ),
                    Td(str(len(job['bars'])))
                )
                for job_id, job in jobs.items()
            ]
        ),
        cls=combine_classes(table, table_sizes.xs, w.full)
    )

@rt("/sse/{job_id}")
def sse_endpoint(job_id: str):
    # Use standard SSE format, not HTMX-specific
    gen = sse_stream(
        monitor,
        job_id,
        interval=0.05,  # Faster updates
        heartbeat=15.0,
        wait_for_start=True,
        start_timeout=10.0
    )
    return StreamingResponse(gen, media_type="text/event-stream")

In [7]:
# Polling endpoint for HTMX (alternative to SSE)
@rt("/poll/{job_id}")
def poll_progress(job_id: str):
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Return updated progress component
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", cls=str(font_size.sm)),
                Progress(value=str(bar.progress), max="100", cls=combine_classes(progress, progress_colors.accent, w.full)),
                cls=str(m.b(2))
            )
        )
    
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(value=str(snapshot['overall_progress']), max="100", cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
        *bars_html,
        # Continue polling if not complete
        Div(
            hx_get=f"/poll/{job_id}",
            hx_trigger="load delay:500ms" if not snapshot['completed'] else "none",
            hx_swap="outerHTML"
        ) if not snapshot['completed'] else Div("Complete!", cls=combine_classes(alert, alert_colors.success)),
        id=f"poll-{job_id}"
    )

In [8]:
# Start server
server = start_test_server(app)
display(HTMX())

In [9]:
# Stop server when done
server.stop()